# NB38 — EX-4 PFC Sync Decision Extraction Proof

**Objective:** Extract the synchronous decision portion (~60 LOC) of the PFC hold
logic from `engine.py` into a pure function in `pfc.py`. The yield loop (Y3 — retry
timeout) stays in engine.py. Only the DECISION moves — "should we keep holding,
escalate, or release?" — not the EXECUTION — "pause the simulation for RETRY_INTERVAL."

## What This Notebook Proves
1. `pfc.py` has **zero SimPy imports** (HC-6)
2. `HoldEvaluation` is a **frozen** dataclass (immutable return type)
3. `evaluate_hold()` passes **unit tests** for all 4 actions (RELEASE, CONTINUE_HOLD, ESCALATE_TO_PFC, HOLD_TIMEOUT)
4. `compute_deterioration()` is **linear, clamped** to [0.0, 1.0] (EP-3 extension point)
5. Full engine regression: toggle OFF vs ON produces **identical** output
6. **DISPOSITION invariant** holds (KL-6)

---
## Cell 1: Imports

In [ ]:
import sys, os

# Resolve src/ whether CWD is project root (VS Code) or notebooks/phase1/ (nbconvert)
for _candidate in [
    os.path.join(os.path.abspath('.'), 'src'),
    os.path.join(os.path.abspath('..'), '..', 'src'),
    os.path.join(os.path.abspath('..'), 'src'),
]:
    if os.path.isdir(_candidate) and _candidate not in sys.path:
        sys.path.insert(0, _candidate)
        break

import importlib

from faer_dev.config.builder import build_engine_from_preset
from faer_dev.decisions.mode import SimulationToggles
from faer_dev.pfc import HoldEvaluation, PFCAction, compute_deterioration, evaluate_hold
print('All imports OK')

---
## Cell 2: Purity + Frozen Check

In [ ]:
# Purity: zero SimPy imports
spec = importlib.util.find_spec("faer_dev.pfc")
assert spec is not None and spec.origin is not None
with open(spec.origin) as f:
    content = f.read()
assert "import simpy" not in content, "FAIL: pfc.py imports simpy"
assert "from simpy" not in content, "FAIL: pfc.py imports from simpy"
print('[PASS] pfc.py has zero SimPy imports (HC-6)')

# Frozen: HoldEvaluation is immutable
result = evaluate_hold(10.0, False, False, 60.0, 240.0)
try:
    result.action = PFCAction.RELEASE
    assert False, "HoldEvaluation should be frozen"
except AttributeError:
    print('[PASS] HoldEvaluation is frozen (immutable dataclass)')

---
## Cell 3: Unit Tests — pure decision function, no SimPy, no engine

In [ ]:
# Test 1: RELEASE when downstream available
r = evaluate_hold(hold_duration=120.0, downstream_available=True,
                  is_pfc_active=False, pfc_threshold=60.0, hold_timeout=240.0)
assert r.action == PFCAction.RELEASE
print(f'[PASS] downstream_available=True  -> {r.action.name}')

# Test 2: CONTINUE_HOLD below threshold
r = evaluate_hold(hold_duration=30.0, downstream_available=False,
                  is_pfc_active=False, pfc_threshold=60.0, hold_timeout=240.0)
assert r.action == PFCAction.CONTINUE_HOLD
print(f'[PASS] hold_duration < threshold  -> {r.action.name}')

# Test 3: ESCALATE_TO_PFC above threshold, not yet PFC
r = evaluate_hold(hold_duration=65.0, downstream_available=False,
                  is_pfc_active=False, pfc_threshold=60.0, hold_timeout=240.0)
assert r.action == PFCAction.ESCALATE_TO_PFC
print(f'[PASS] hold_duration > threshold  -> {r.action.name}')

# Test 4: CONTINUE_HOLD when already PFC (no re-escalation)
r = evaluate_hold(hold_duration=100.0, downstream_available=False,
                  is_pfc_active=True, pfc_threshold=60.0, hold_timeout=240.0)
assert r.action == PFCAction.CONTINUE_HOLD
print(f'[PASS] already PFC, below timeout -> {r.action.name}')

# Test 5: HOLD_TIMEOUT when exceeded
r = evaluate_hold(hold_duration=250.0, downstream_available=False,
                  is_pfc_active=True, pfc_threshold=60.0, hold_timeout=240.0)
assert r.action == PFCAction.HOLD_TIMEOUT
assert r.should_emit_timeout is True
print(f'[PASS] hold_duration > timeout    -> {r.action.name}')

# Test 6: First check flags hold_start emission
r = evaluate_hold(hold_duration=0.0, downstream_available=False,
                  is_pfc_active=False, pfc_threshold=60.0, hold_timeout=240.0,
                  is_first_check=True)
assert r.action == PFCAction.CONTINUE_HOLD
assert r.should_emit_hold_start is True
print(f'[PASS] first check               -> {r.action.name} + emit_hold_start')

print()
print('All 6 PFC unit tests pass — no SimPy, no engine, pure function')

---
## Cell 4: Deterioration Model (EP-3 extension point)

In [ ]:
# Linear deterioration: severity + hold_duration * base_rate
new_sev = compute_deterioration(0.5, hold_duration=60.0, base_rate=0.005)
assert abs(new_sev - 0.8) < 0.001
print(f'[PASS] Linear deterioration: 0.5 + 60*0.005 = {new_sev}')

# Clamped to [0, 1]
new_sev = compute_deterioration(0.9, hold_duration=200.0, base_rate=0.01)
assert new_sev == 1.0
print(f'[PASS] Clamped to 1.0: 0.9 + 200*0.01 -> {new_sev}')

# Zero duration = no change
new_sev = compute_deterioration(0.3, hold_duration=0.0, base_rate=0.01)
assert new_sev == 0.3
print(f'[PASS] Zero duration: severity unchanged at {new_sev}')

---
## Cell 5: Full Engine Regression — all toggles OFF vs ON

In [ ]:
ALL_OFF = SimulationToggles(
    enable_extracted_routing=False, enable_extracted_metrics=False,
    enable_typed_emitter=False, enable_extracted_pfc=False,
)
ALL_ON = SimulationToggles(
    enable_extracted_routing=True, enable_extracted_metrics=True,
    enable_typed_emitter=True, enable_extracted_pfc=True,
)

def run_engine(toggles, seed=42, max_patients=20):
    engine = build_engine_from_preset("coin", seed=seed, toggles=toggles)
    metrics = engine.run(duration=600.0, max_patients=max_patients)
    return metrics, engine.events

m_off, events_off = run_engine(ALL_OFF)
m_on, events_on = run_engine(ALL_ON)

assert m_off["total_arrivals"] == m_on["total_arrivals"]
assert m_off["completed"] == m_on["completed"]
assert m_off["outcomes"] == m_on["outcomes"]
print(f'[PASS] ALL toggles OFF vs ON: metrics identical')
print(f'       arrivals={m_on["total_arrivals"]}, completed={m_on["completed"]}')

---
## Cell 6: DISPOSITION Invariant (KL-6)

In [ ]:
arrivals = sum(1 for e in events_on if e["type"] == "ARRIVAL")
dispositions = sum(1 for e in events_on if e["type"] == "DISPOSITION")
in_system = m_on["in_system"]
assert arrivals == dispositions + in_system, \
    f"KL-6 VIOLATION: {arrivals} != {dispositions} + {in_system}"
assert arrivals > 0
print(f'[PASS] DISPOSITION invariant (KL-6): {arrivals} arrivals == {dispositions} dispositions + {in_system} in_system')

---
## Summary

| Check | Status |
|-------|--------|
| `pfc.py` has zero SimPy imports (HC-6) | Verified |
| `HoldEvaluation` is frozen (immutable) | Verified |
| RELEASE when downstream available | Verified |
| CONTINUE_HOLD below threshold | Verified |
| ESCALATE_TO_PFC above threshold | Verified |
| HOLD_TIMEOUT when exceeded | Verified |
| First check emits hold_start | Verified |
| Deterioration: linear, clamped [0,1] (EP-3) | Verified |
| Full engine regression: ALL OFF vs ON | Verified |
| DISPOSITION invariant (KL-6) | Verified |

**Conclusion:** `pfc.py` extraction is correct. Y3 stays in engine.py. Decision logic is pure and testable.